# ⭐ Day 87: Object Detection with YOLO - Real-Time Detection & Custom Training
## Day 87 of 369-day Python & AI Learning Path

Welcome to Day 87! Today we advance into **Object Detection**, one of the most powerful applications of Computer Vision. We'll master **YOLO (You Only Look Once)** models for real-time detection and custom training. 🚀

## 📋 Table of Contents

1. [Introduction to Object Detection vs Image Classification](#1)
2. [Understanding YOLO Architecture](#2)
3. [Loading Pre-trained YOLO Model](#3)
4. [Real-Time Object Detection on Images](#4)
5. [Real-Time Object Detection on Videos](#5)
6. [Custom Dataset Preparation](#6)
7. [Fine-tuning YOLO on Custom Dataset](#7)
8. [Model Evaluation Metrics](#8)
9. [Deployment Ideas](#9)
10. [Hands-On Exercises](#10)
11. [Solutions](#11)
12. [Summary & Day 88 Teaser](#12)

## 1. Introduction to Object Detection vs Image Classification 🧠 <a id='1'></a>

**Image Classification** assigns a single label to an entire image. **Object Detection** goes further—it identifies *what* objects are present, *where* they are, and draws bounding boxes around them.

| Task | Output | Use Case |
|------|--------|----------|
| Classification | Single label | Is this a cat or dog? |
| Object Detection | Bounding boxes + labels + confidence | Where are the cats and dogs in this image? |
| Instance Segmentation | Pixel-level masks | Exact shape of each cat and dog |

Object Detection powers: autonomous vehicles, surveillance systems, medical imaging, retail analytics, and robotics!

## 2. Understanding YOLO Architecture 🚀 <a id='2'></a>

**YOLO (You Only Look Once)** revolutionized object detection by treating it as a single regression problem. Instead of sliding windows or region proposals, YOLO divides the image into a grid and predicts bounding boxes and class probabilities simultaneously.

### Key YOLO Versions:
- **YOLOv5**: Ultralytics, PyTorch-based, excellent balance of speed/accuracy
- **YOLOv8**: Latest architecture, improved backbone, anchor-free detection, better accuracy

### YOLO Architecture Components:
1. **Backbone** (CSPDarknet): Feature extraction
2. **Neck** (PANet): Feature fusion across scales
3. **Head**: Prediction of bounding boxes, objectness, and class probabilities

### Output Format:
Each detection returns: `[x_center, y_center, width, height, confidence, class_probabilities]`

In [ ]:
# Install required libraries
!pip install ultralytics opencv-python matplotlib seaborn -q

import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries installed successfully!")

## 3. Loading Pre-trained YOLO Model 💡 <a id='3'></a>

We'll use the `ultralytics` library which provides easy access to YOLOv8 models. Pre-trained models are available in multiple sizes: `nano`, `small`, `medium`, `large`, `extra-large`.

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

# Load pre-trained YOLOv8n (nano) model - fastest, good for real-time
model = YOLO('yolov8n.pt')

# Display model info
print("🧠 Model loaded: YOLOv8n (Nano)")
print(f"📊 Model parameters: {sum(p.numel() for p in model.model.parameters()):,}")
print(f"🏷️ Number of classes: {len(model.names)}")
print(f"📝 Class names (first 10): {list(model.names.values())[:10]}")

## 4. Real-Time Object Detection on Images 📸 <a id='4'></a>

Let's perform object detection on sample images and visualize the results with bounding boxes and confidence scores.

In [ ]:
# Create a sample image for demonstration (street scene simulation)
# In practice, you would load real images
def create_sample_image():
    """Create a sample image with simple shapes to simulate objects"""
    img = np.ones((480, 640, 3), dtype=np.uint8) * 240  # Light gray background
    
    # Simulate a 'person' (blue rectangle)
    cv2.rectangle(img, (100, 150), (180, 400), (200, 100, 50), -1)
    cv2.rectangle(img, (100, 150), (180, 400), (0, 0, 0), 2)
    
    # Simulate a 'car' (red rectangle)
    cv2.rectangle(img, (300, 280), (500, 380), (50, 50, 200), -1)
    cv2.rectangle(img, (300, 280), (500, 380), (0, 0, 0), 2)
    
    # Simulate a 'dog' (green rectangle)
    cv2.rectangle(img, (450, 320), (550, 420), (50, 200, 50), -1)
    cv2.rectangle(img, (450, 320), (550, 420), (0, 0, 0), 2)
    
    return img

sample_img = create_sample_image()

# Perform detection
results = model(sample_img)

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Original image
axes[0].imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
axes[0].axis('off')

# Annotated image
annotated_img = results[0].plot()
axes[1].imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
axes[1].set_title('YOLO Detection Results', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Print detection details
print("\n📊 Detection Results:")
for r in results:
    boxes = r.boxes
    for box in boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        class_name = model.names[cls_id]
        print(f"  🏷️ {class_name}: {conf:.2%} confidence")

In [ ]:
# Detection on multiple images with confidence filtering
def detect_and_visualize(image, conf_threshold=0.5, title="Detection"):
    """Perform detection and visualize with custom confidence threshold"""
    results = model(image, conf=conf_threshold)
    annotated = results[0].plot()
    
    # Count detections per class
    detections = {}
    for box in results[0].boxes:
        cls_name = model.names[int(box.cls[0])]
        detections[cls_name] = detections.get(cls_name, 0) + 1
    
    return annotated, detections, results

# Create multiple test scenarios
test_images = []
titles = []

# Image 1: Dense scene
img1 = np.ones((480, 640, 3), dtype=np.uint8) * 220
cv2.rectangle(img1, (50, 100), (120, 350), (100, 150, 200), -1)   # person
cv2.rectangle(img1, (200, 120), (280, 340), (100, 150, 200), -1)  # person
cv2.rectangle(img1, (400, 250), (580, 380), (50, 80, 180), -1)    # car
cv2.rectangle(img1, (300, 300), (380, 400), (80, 160, 80), -1)    # dog
test_images.append(img1)
titles.append("Scene 1: Street View")

# Image 2: Single prominent object
img2 = np.ones((480, 640, 3), dtype=np.uint8) * 200
cv2.rectangle(img2, (200, 100), (450, 400), (120, 120, 120), -1)  # large object
test_images.append(img2)
titles.append("Scene 2: Single Object")

# Image 3: Small objects
img3 = np.ones((480, 640, 3), dtype=np.uint8) * 230
for i in range(5):
    x = 100 + i * 100
    cv2.rectangle(img3, (x, 200), (x+60, 320), (150, 100, 50), -1)
test_images.append(img3)
titles.append("Scene 3: Multiple Small Objects")

# Visualize all
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (img, title) in enumerate(zip(test_images, titles)):
    # Original
    axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f'{title}\n(Original)', fontsize=11, fontweight='bold')
    axes[idx].axis('off')
    
    # Detection
    annotated, dets, _ = detect_and_visualize(img, conf_threshold=0.3)
    axes[idx + 3].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    det_text = "\n".join([f"{k}: {v}" for k, v in dets.items()]) if dets else "No detections"
    axes[idx + 3].set_title(f'{title}\nDetected: {det_text}', fontsize=11, fontweight='bold')
    axes[idx + 3].axis('off')

plt.suptitle('🎯 YOLO Object Detection on Multiple Scenes', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Real-Time Object Detection on Videos 📹 <a id='5'></a>

YOLO excels at real-time video processing. Let's simulate video frame processing and visualize detection over time.

In [ ]:
# Simulate video processing with frame-by-frame detection
def process_video_frames(num_frames=8):
    """Simulate video by creating moving objects across frames"""
    frames = []
    detections_history = []
    
    for i in range(num_frames):
        # Create frame with moving object
        frame = np.ones((480, 640, 3), dtype=np.uint8) * 200
        
        # Moving car (simulates motion)
        x_pos = 50 + i * 60
        cv2.rectangle(frame, (x_pos, 250), (x_pos + 150, 350), (50, 50, 200), -1)
        cv2.rectangle(frame, (x_pos, 250), (x_pos + 150, 350), (0, 0, 0), 2)
        
        # Static person
        cv2.rectangle(frame, (400, 150), (470, 400), (200, 100, 50), -1)
        cv2.rectangle(frame, (400, 150), (470, 400), (0, 0, 0), 2)
        
        # Run detection
        results = model(frame, conf=0.3)
        annotated = results[0].plot()
        
        # Count objects
        frame_dets = {}
        for box in results[0].boxes:
            cls_name = model.names[int(box.cls[0])]
            frame_dets[cls_name] = frame_dets.get(cls_name, 0) + 1
        
        frames.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        detections_history.append(frame_dets)
    
    return frames, detections_history

video_frames, det_history = process_video_frames(num_frames=8)

# Display video frames
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, (frame, dets) in enumerate(zip(video_frames, det_history)):
    axes[i].imshow(frame)
    det_text = " | ".join([f"{k}:{v}" for k, v in dets.items()]) if dets else "None"
    axes[i].set_title(f'Frame {i+1}\nObjects: {det_text}', fontsize=10, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('📹 Real-Time Video Object Detection (Simulated)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Video processing simulation complete!")
print(f"📊 Processed {len(video_frames)} frames with real-time detection.")

In [ ]:
# FPS Benchmark Simulation
import time

def benchmark_fps(num_iterations=50):
    """Benchmark detection speed"""
    test_img = np.ones((480, 640, 3), dtype=np.uint8) * 200
    cv2.rectangle(test_img, (200, 200), (400, 400), (100, 100, 100), -1)
    
    # Warm-up
    for _ in range(5):
        _ = model(test_img, verbose=False)
    
    # Benchmark
    start_time = time.time()
    for _ in range(num_iterations):
        _ = model(test_img, verbose=False)
    end_time = time.time()
    
    total_time = end_time - start_time
    fps = num_iterations / total_time
    ms_per_frame = (total_time / num_iterations) * 1000
    
    return fps, ms_per_frame

fps, ms_frame = benchmark_fps(30)

print(f"🚀 YOLOv8n Performance Benchmark:")
print(f"   ⚡ FPS: {fps:.1f}")
print(f"   ⏱️  ms/frame: {ms_frame:.1f}")
print(f"   💡 Status: {'Real-time capable! ✅' if fps > 15 else 'May need optimization'}")

## 6. Custom Dataset Preparation Concepts 🗂️ <a id='6'></a>

Training YOLO on custom data requires careful dataset preparation. Here's the standard structure:

```
dataset/
├── train/
│   ├── images/          # Training images (.jpg, .png)
│   └── labels/          # YOLO format labels (.txt)
├── valid/
│   ├── images/          # Validation images
│   └── labels/          # Validation labels
├── test/
│   ├── images/          # Test images
│   └── labels/          # Test labels
└── data.yaml            # Dataset configuration
```

### YOLO Label Format (`.txt` per image):
Each line represents one object: `class_id x_center y_center width height`
- All values are normalized to [0, 1] relative to image dimensions
- `class_id` starts from 0

### data.yaml Example:
```yaml
path: /path/to/dataset
train: train/images
val: valid/images
test: test/images

nc: 3  # number of classes
names: ['person', 'car', 'dog']
```

In [ ]:
# Demonstrate label format conversion
def create_example_label_file():
    """Create example YOLO format labels"""
    
    # Example: Image 640x480 with 3 objects
    # Object 1: Person at center (class 0)
    # Object 2: Car at right (class 1)  
    # Object 3: Dog at bottom-left (class 2)
    
    labels = [
        "0 0.500000 0.500000 0.200000 0.400000",  # person
        "1 0.750000 0.600000 0.300000 0.200000",  # car
        "2 0.250000 0.800000 0.150000 0.150000",  # dog
    ]
    
    print("📝 Example YOLO Label Format (.txt file):")
    print("-" * 50)
    for i, label in enumerate(labels, 1):
        parts = label.split()
        print(f"Object {i}: class={parts[0]}, center=({parts[1]}, {parts[2]}), size=({parts[3]}, {parts[4]})")
    print("-" * 50)
    print("\n💡 Note: All coordinates are normalized [0, 1]")
    
    return labels

example_labels = create_example_label_file()

# Visualize bounding box coordinates
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_title('📐 YOLO Bounding Box Coordinate System', fontsize=14, fontweight='bold')
ax.set_xlabel('X (normalized)', fontsize=12)
ax.set_ylabel('Y (normalized)', fontsize=12)

# Draw example boxes
boxes = [
    (0.5, 0.5, 0.2, 0.4, 'Person (cls 0)', 'blue'),
    (0.75, 0.6, 0.3, 0.2, 'Car (cls 1)', 'red'),
    (0.25, 0.8, 0.15, 0.15, 'Dog (cls 2)', 'green')
]

for x, y, w, h, label, color in boxes:
    rect = plt.Rectangle((x - w/2, y - h/2), w, h, 
                          linewidth=2, edgecolor=color, facecolor=color, alpha=0.3)
    ax.add_patch(rect)
    ax.plot(x, y, 'ko', markersize=6)
    ax.annotate(f'{label}\ncenter=({x},{y})', (x, y), 
                textcoords="offset points", xytext=(10, 10), fontsize=9)

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Fine-tuning YOLO on a Custom Dataset 🔧 <a id='7'></a>

Let's simulate the fine-tuning process. In practice, you would have a labeled dataset. We'll demonstrate the training code structure and visualize training curves.

In [ ]:
# Simulate training process and generate training curves
# In practice: model.train(data='data.yaml', epochs=100, imgsz=640)

np.random.seed(87)
epochs = 100

# Simulate realistic training metrics
def simulate_training_curves(epochs=100):
    """Generate realistic training curves"""
    
    # Box loss (decreasing)
    box_loss = 0.08 * np.exp(-np.linspace(0, 3, epochs)) + 0.015 + np.random.normal(0, 0.002, epochs)
    box_loss_val = box_loss + 0.005 + np.random.normal(0, 0.001, epochs)
    
    # Classification loss (decreasing)
    cls_loss = 0.5 * np.exp(-np.linspace(0, 2.5, epochs)) + 0.02 + np.random.normal(0, 0.005, epochs)
    cls_loss_val = cls_loss + 0.01 + np.random.normal(0, 0.003, epochs)
    
    # DFL loss (decreasing)
    dfl_loss = 0.06 * np.exp(-np.linspace(0, 3, epochs)) + 0.01 + np.random.normal(0, 0.001, epochs)
    
    # mAP metrics (increasing)
    map50 = 1 - 0.7 * np.exp(-np.linspace(0, 2.5, epochs)) + np.random.normal(0, 0.01, epochs)
    map50_95 = 1 - 0.85 * np.exp(-np.linspace(0, 2.5, epochs)) + np.random.normal(0, 0.008, epochs)
    
    # Precision & Recall
    precision = 1 - 0.6 * np.exp(-np.linspace(0, 2, epochs)) + np.random.normal(0, 0.01, epochs)
    recall = 1 - 0.65 * np.exp(-np.linspace(0, 2.2, epochs)) + np.random.normal(0, 0.01, epochs)
    
    return {
        'box_loss': np.clip(box_loss, 0.01, 0.1),
        'box_loss_val': np.clip(box_loss_val, 0.01, 0.1),
        'cls_loss': np.clip(cls_loss, 0.01, 0.5),
        'cls_loss_val': np.clip(cls_loss_val, 0.01, 0.5),
        'dfl_loss': np.clip(dfl_loss, 0.005, 0.07),
        'mAP50': np.clip(map50, 0.3, 0.98),
        'mAP50-95': np.clip(map50_95, 0.2, 0.85),
        'precision': np.clip(precision, 0.4, 0.98),
        'recall': np.clip(recall, 0.4, 0.98)
    }

metrics = simulate_training_curves(epochs)

# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Loss curves
ax1 = axes[0, 0]
ax1.plot(metrics['box_loss'], label='Box Loss (train)', color='blue', linewidth=2)
ax1.plot(metrics['box_loss_val'], label='Box Loss (val)', color='blue', linestyle='--', alpha=0.7)
ax1.plot(metrics['cls_loss'], label='Cls Loss (train)', color='red', linewidth=2)
ax1.plot(metrics['cls_loss_val'], label='Cls Loss (val)', color='red', linestyle='--', alpha=0.7)
ax1.plot(metrics['dfl_loss'], label='DFL Loss', color='green', linewidth=2)
ax1.set_title('📉 Training & Validation Losses', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# mAP curves
ax2 = axes[0, 1]
ax2.plot(metrics['mAP50'], label='mAP@0.5', color='purple', linewidth=2.5)
ax2.plot(metrics['mAP50-95'], label='mAP@0.5:0.95', color='orange', linewidth=2.5)
ax2.set_title('📈 Mean Average Precision (mAP)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('mAP Score')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

# Precision & Recall
ax3 = axes[1, 0]
ax3.plot(metrics['precision'], label='Precision', color='teal', linewidth=2.5)
ax3.plot(metrics['recall'], label='Recall', color='coral', linewidth=2.5)
ax3.set_title('🎯 Precision & Recall Over Training', fontsize=13, fontweight='bold')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Score')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_ylim(0, 1)

# Final metrics summary
ax4 = axes[1, 1]
ax4.axis('off')
final_metrics = {
    'mAP@0.5': f"{metrics['mAP50'][-1]:.3f}",
    'mAP@0.5:0.95': f"{metrics['mAP50-95'][-1]:.3f}",
    'Precision': f"{metrics['precision'][-1]:.3f}",
    'Recall': f"{metrics['recall'][-1]:.3f}",
    'Final Box Loss': f"{metrics['box_loss'][-1]:.4f}",
    'Final Cls Loss': f"{metrics['cls_loss'][-1]:.4f}"
}

summary_text = "🏆 Final Training Results\n" + "="*30 + "\n\n"
for metric, value in final_metrics.items():
    summary_text += f"{metric:20s}: {value}\n"

ax4.text(0.1, 0.5, summary_text, fontsize=13, family='monospace', 
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax4.set_title('📊 Final Metrics Summary', fontsize=13, fontweight='bold')

plt.suptitle('🚀 YOLO Custom Training Visualization', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("✅ Training simulation complete! In practice, use:")
print("   model.train(data='custom_data.yaml', epochs=100, imgsz=640, batch=16)")

In [ ]:
# Training code template (commented for reference)
training_template = '''
# 🚀 YOLO Custom Training Template

from ultralytics import YOLO

# 1. Load pre-trained model (transfer learning)
model = YOLO('yolov8n.pt')  # or 'yolov8s.pt', 'yolov8m.pt'

# 2. Train on custom dataset
results = model.train(
    data='path/to/data.yaml',    # Dataset config
    epochs=100,                   # Training epochs
    imgsz=640,                    # Image size
    batch=16,                     # Batch size
    workers=8,                    # Data loading workers
    device=0,                     # GPU device
    patience=20,                  # Early stopping patience
    save=True,                    # Save best model
    project='runs/detect',        # Project directory
    name='custom_training'        # Run name
)

# 3. Validate
metrics = model.val()

# 4. Export for deployment
model.export(format='onnx')     # ONNX format
model.export(format='torchscript')  # TorchScript
'''

print(training_template)

## 8. Model Evaluation Metrics 📊 <a id='8'></a>

Understanding evaluation metrics is crucial for assessing object detection performance.

### Key Metrics:
- **IoU (Intersection over Union)**: Measures overlap between predicted and ground-truth boxes
- **Precision**: TP / (TP + FP) — accuracy of positive predictions
- **Recall**: TP / (TP + FN) — coverage of actual positives
- **mAP@0.5**: Mean Average Precision at IoU threshold 0.5
- **mAP@0.5:0.95**: Average mAP across IoU thresholds 0.5 to 0.95 (COCO standard)

### Confusion Matrix:
Shows classification performance across all classes.

In [ ]:
# Generate and visualize confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Simulate predictions vs ground truth for 5 classes
np.random.seed(87)
classes = ['person', 'car', 'dog', 'cat', 'bicycle']
n_classes = len(classes)

# Create realistic confusion matrix (high diagonal = good predictions)
true_labels = []
pred_labels = []

for i, cls in enumerate(classes):
    n_samples = np.random.randint(80, 150)
    # 85% correct predictions
    n_correct = int(n_samples * 0.85)
    n_incorrect = n_samples - n_correct
    
    true_labels.extend([i] * n_samples)
    pred_labels.extend([i] * n_correct)
    
    # Distribute incorrect predictions among other classes
    other_classes = [j for j in range(n_classes) if j != i]
    for _ in range(n_incorrect):
        pred_labels.append(np.random.choice(other_classes))

cm = confusion_matrix(true_labels, pred_labels, labels=range(n_classes))

# Normalize by row (recall per class)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title('📊 Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title('📊 Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.suptitle('🎯 Object Detection Confusion Matrix', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate per-class metrics
print("\n📈 Per-Class Performance:")
print("-" * 50)
for i, cls in enumerate(classes):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"{cls:12s}: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")

In [ ]:
# IoU Visualization
def plot_iou_visualization():
    """Visualize Intersection over Union concept"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    scenarios = [
        ("Perfect Match\n(IoU = 1.0)", 0.3, 0.3, 0.4, 0.4, 0.3, 0.3, 0.4, 0.4),
        ("Partial Overlap\n(IoU = 0.44)", 0.2, 0.2, 0.5, 0.5, 0.35, 0.35, 0.5, 0.5),
        ("Poor Overlap\n(IoU = 0.08)", 0.1, 0.1, 0.3, 0.3, 0.6, 0.6, 0.3, 0.3)
    ]
    
    colors_gt = 'lightblue'
    colors_pred = 'lightcoral'
    
    for ax, (title, gx, gy, gw, gh, px, py, pw, ph) in zip(axes, scenarios):
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect('equal')
        
        # Ground truth box
        gt = plt.Rectangle((gx, gy), gw, gh, linewidth=2, 
                           edgecolor='blue', facecolor=colors_gt, alpha=0.5, label='Ground Truth')
        ax.add_patch(gt)
        
        # Prediction box
        pred = plt.Rectangle((px, py), pw, ph, linewidth=2,
                             edgecolor='red', facecolor=colors_pred, alpha=0.5, label='Prediction')
        ax.add_patch(pred)
        
        # Calculate IoU
        x1 = max(gx, px)
        y1 = max(gy, py)
        x2 = min(gx + gw, px + pw)
        y2 = min(gy + gh, py + ph)
        
        if x2 > x1 and y2 > y1:
            inter = (x2 - x1) * (y2 - y1)
            union = gw * gh + pw * ph - inter
            iou = inter / union
            
            # Draw intersection
            inter_rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                                       facecolor='purple', alpha=0.7)
            ax.add_patch(inter_rect)
        
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('📐 Intersection over Union (IoU) Visualization', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_iou_visualization()

print("\n💡 IoU Thresholds:")
print("   • IoU ≥ 0.5  → Positive detection (PASCAL VOC standard)")
print("   • IoU ≥ 0.75 → Strict detection (COCO AP75)")
print("   • IoU ≥ 0.9  → Very strict, near-perfect localization")

## 9. Deployment Ideas for Real-Time Applications 🌐 <a id='9'></a>

YOLO models can be deployed in various production environments:

### 💻 Edge Devices
- **NVIDIA Jetson**: Optimized TensorRT inference
- **Raspberry Pi**: Lightweight YOLOv8n with NCNN
- **Coral TPU**: Quantized models for ultra-low latency

### ☁️ Cloud Deployment
- **FastAPI/Flask**: REST API for image/video inference
- **Docker**: Containerized deployment with GPU support
- **AWS/GCP/Azure**: Scalable inference endpoints

### 📱 Mobile & Web
- **ONNX Runtime**: Cross-platform deployment
- **TensorFlow.js**: Browser-based detection
- **Core ML**: iOS native inference

### 🏭 Industry Applications
| Domain | Application |
|--------|-------------|
| Manufacturing | Defect detection on production lines |
| Retail | Shelf monitoring & inventory management |
| Healthcare | Tumor detection in medical imaging |
| Agriculture | Crop disease & pest detection |
| Security | Intrusion detection & crowd monitoring |
| Automotive | Pedestrian & vehicle detection for ADAS |

In [ ]:
# Deployment-ready inference function
class YOLODeployer:
","    """Production-ready YOLO inference wrapper"""
    
    def __init__(self, model_path='yolov8n.pt', conf_threshold=0.5, device='cpu'):
        self.model = YOLO(model_path)
        self.conf_threshold = conf_threshold
        self.device = device
        self.class_names = self.model.names
        print(f"🚀 Deployer initialized with {model_path}")
    
    def predict_image(self, image_path):
        """Run inference on a single image"""
        results = self.model(image_path, conf=self.conf_threshold, device=self.device)
        return self._format_results(results[0])
    
    def predict_batch(self, image_paths, batch_size=8):
        """Run inference on a batch of images"""
        all_results = []
        for i in range(0, len(image_paths), batch_size):
            batch = image_paths[i:i+batch_size]
            results = self.model(batch, conf=self.conf_threshold, device=self.device)
            all_results.extend([self._format_results(r) for r in results])
        return all_results
    
    def _format_results(self, result):
        """Format detection results into structured output"""
        detections = []
        for box in result.boxes:
            detections.append({
                'class': self.class_names[int(box.cls[0])],
                'confidence': float(box.conf[0]),
                'bbox': box.xyxy[0].tolist()  # [x1, y1, x2, y2]
            })
        return {
            'num_detections': len(detections),
            'detections': detections
        }
    
    def export_model(self, format='onnx'):
        """Export model for deployment"""
        path = self.model.export(format=format)
        print(f"✅ Model exported to: {path}")
        return path

# Initialize deployer
deployer = YOLODeployer(conf_threshold=0.4)

# Example inference
test_image = create_sample_image()
results = deployer.predict_image(test_image)

print("\n📊 Structured Output:")
print(f"Objects detected: {results['num_detections']}")
for det in results['detections']:
    print(f"  • {det['class']}: {det['confidence']:.2%} at {det['bbox']}")

print("\n💡 Ready for production deployment!")

## 🛠️ Hands-On Exercises <a id='10'></a>

Test your understanding with these practical challenges!

### Exercise 1: 🔍 Confidence Threshold Tuning
Load the YOLOv8n model and test it on an image with different confidence thresholds (0.1, 0.3, 0.5, 0.7, 0.9). Visualize how the number and quality of detections change. What is the optimal threshold for your use case?

### Exercise 2: 🎨 Custom Class Filter
Create a function that runs YOLO detection but only returns detections for specific classes (e.g., only 'person' and 'car'). Filter out all other detections and draw bounding boxes in different colors for each allowed class.

### Exercise 3: 📏 Bounding Box Area Analysis
Write a function that calculates the area of each detected bounding box (in pixels). Classify detections as 'small', 'medium', or 'large' based on area thresholds you define. Print statistics showing the distribution of object sizes in an image.

### Exercise 4: 🔄 Video Processing Pipeline
Build a complete video processing pipeline that reads frames, runs YOLO detection, counts unique objects per frame, and generates a time-series plot showing how object counts change across frames. Add a feature to save the annotated video to disk.

## ✅ Solutions <a id='11'></a>

Complete solutions for all exercises are provided below. Study them carefully and compare with your implementations!

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ✅ SOLUTION 1: Confidence Threshold Tuning
# ═══════════════════════════════════════════════════════════════

def exercise_1_solution():
    """Demonstrate effect of different confidence thresholds"""
    
    # Create test image with objects at varying clarity
    img = np.ones((480, 640, 3), dtype=np.uint8) * 210
    
    # Clear objects (high confidence expected)
    cv2.rectangle(img, (100, 150), (180, 400), (200, 100, 50), -1)   # person
    cv2.rectangle(img, (300, 250), (500, 380), (50, 50, 200), -1)    # car
    
    # Less clear object (lower confidence possible)
    cv2.rectangle(img, (520, 300), (600, 420), (100, 100, 100), -1)  # ambiguous
    
    thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, conf in enumerate(thresholds):
        results = model(img, conf=conf)
        annotated = results[0].plot()
        
        axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        count = len(results[0].boxes)
        axes[idx].set_title(f'conf={conf}\nDetections: {count}', fontsize=11, fontweight='bold')
        axes[idx].axis('off')
    
    # Summary plot
    axes[5].axis('off')
    summary = "📊 Threshold Impact:\n\n"
    summary += "• Low (0.1): More detections,\n  more false positives\n\n"
    summary += "• Medium (0.5): Balanced\n  precision-recall\n\n"
    summary += "• High (0.9): Fewer detections,\n  high precision only"
    axes[5].text(0.1, 0.5, summary, fontsize=12, verticalalignment='center',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    
    plt.suptitle('✅ Solution 1: Confidence Threshold Analysis', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

exercise_1_solution()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ✅ SOLUTION 2: Custom Class Filter
# ═══════════════════════════════════════════════════════════════

def exercise_2_solution():
    """Filter detections by specific classes with custom colors"""
    
    def detect_filtered(image, allowed_classes, model):
        """Run detection and filter by allowed classes"""
        results = model(image)
        
        # Color map for allowed classes
        color_map = {
            'person': (255, 0, 0),      # Red
            'car': (0, 255, 0),          # Green
            'dog': (0, 0, 255),          # Blue
            'cat': (255, 255, 0),        # Yellow
            'bicycle': (255, 0, 255)     # Magenta
        }
        
        # Create annotated image manually
        annotated = image.copy()
        detections = []
        
        for box in results[0].boxes:
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            conf = float(box.conf[0])
            
            if cls_name in allowed_classes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                color = color_map.get(cls_name, (128, 128, 128))
                
                # Draw bounding box
                cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
                
                # Draw label background
                label = f"{cls_name} {conf:.2f}"
                (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                cv2.rectangle(annotated, (x1, y1-text_h-10), (x1+text_w, y1), color, -1)
                cv2.putText(annotated, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 
                           0.5, (255, 255, 255), 2)
                
                detections.append({'class': cls_name, 'confidence': conf})
        
        return annotated, detections
    
    # Test image
    test_img = np.ones((480, 640, 3), dtype=np.uint8) * 220
    cv2.rectangle(test_img, (80, 140), (160, 390), (200, 100, 50), -1)   # person
    cv2.rectangle(test_img, (280, 240), (480, 370), (50, 50, 200), -1)   # car
    cv2.rectangle(test_img, (500, 290), (580, 410), (50, 200, 50), -1)   # dog
    cv2.rectangle(test_img, (350, 100), (420, 230), (200, 200, 50), -1)  # cat
    
    # Run with different filters
    filters = [
        ['person', 'car'],
        ['person', 'dog', 'cat'],
        ['car'],
        ['person', 'car', 'dog', 'cat']
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, allowed in enumerate(filters):
        annotated, dets = detect_filtered(test_img, allowed, model)
        axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        det_text = "\n".join([f"• {d['class']}: {d['confidence']:.2f}" for d in dets])
        axes[idx].set_title(f'Allowed: {allowed}\n{det_text}', fontsize=10, fontweight='bold')
        axes[idx].axis('off')
    
    plt.suptitle('✅ Solution 2: Custom Class Filtering', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

exercise_2_solution()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ✅ SOLUTION 3: Bounding Box Area Analysis
# ═══════════════════════════════════════════════════════════════

def exercise_3_solution():
    """Analyze bounding box areas and classify by size"""
    
    def analyze_object_sizes(image, model, small_thresh=5000, large_thresh=50000):
        """Detect objects and classify by bounding box area"""
        results = model(image)
        
        size_categories = {'small': [], 'medium': [], 'large': []}
        all_areas = []
        
        annotated = image.copy()
        
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            area = (x2 - x1) * (y2 - y1)
            cls_name = model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            
            all_areas.append(area)
            
            # Classify by area
            if area < small_thresh:
                category = 'small'
                color = (0, 255, 255)  # Cyan
            elif area < large_thresh:
                category = 'medium'
                color = (0, 165, 255)  # Orange
            else:
                category = 'large'
                color = (0, 0, 255)    # Red
            
            size_categories[category].append({
                'class': cls_name,
                'area': area,
                'confidence': conf
            })
            
            # Draw box with size-based color
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            label = f"{cls_name} | {area}px²"
            cv2.putText(annotated, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX,
                       0.4, color, 1)
        
        return annotated, size_categories, all_areas
    
    # Create image with varied object sizes
    test_img = np.ones((600, 800, 3), dtype=np.uint8) * 230
    
    # Large object
    cv2.rectangle(test_img, (50, 50), (350, 450), (200, 100, 50), -1)
    # Medium object
    cv2.rectangle(test_img, (450, 100), (650, 350), (50, 50, 200), -1)
    # Small object
    cv2.rectangle(test_img, (700, 400), (780, 480), (50, 200, 50), -1)
    # Another medium
    cv2.rectangle(test_img, (400, 450), (550, 580), (200, 200, 50), -1)
    
    annotated, categories, areas = analyze_object_sizes(test_img, model, 
                                                        small_thresh=8000, 
                                                        large_thresh=80000)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Annotated image
    axes[0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[0].set_title('📸 Size-Classified Detections\n(Cyan=Small, Orange=Medium, Red=Large)', 
                      fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Statistics
    axes[1].axis('off')
    stats_text = "📊 Object Size Analysis\n" + "="*35 + "\n\n"
    
    for cat in ['small', 'medium', 'large']:
        objs = categories[cat]
        stats_text += f"🟡 {cat.upper()} objects ({len(objs)}):\n"
        for obj in objs:
            stats_text += f"   • {obj['class']}: {obj['area']:,} px²\n"
        stats_text += "\n"
    
    if areas:
        stats_text += f"📈 Statistics:\n"
        stats_text += f"   Total objects: {len(areas)}\n"
        stats_text += f"   Mean area: {np.mean(areas):,.0f} px²\n"
        stats_text += f"   Median area: {np.median(areas):,.0f} px²\n"
        stats_text += f"   Max area: {max(areas):,} px²\n"
        stats_text += f"   Min area: {min(areas):,} px²"
    
    axes[1].text(0.05, 0.95, stats_text, fontsize=11, family='monospace',
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    
    plt.suptitle('✅ Solution 3: Bounding Box Area Analysis', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

exercise_3_solution()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ✅ SOLUTION 4: Video Processing Pipeline
# ═══════════════════════════════════════════════════════════════

def exercise_4_solution():
    """Complete video processing pipeline with time-series analysis"""
    
    def process_video_pipeline(num_frames=20):
        """Process simulated video and track objects over time"""
        
        frame_data = []
        annotated_frames = []
        
        for frame_idx in range(num_frames):
            # Create frame with dynamic scene
            frame = np.ones((480, 640, 3), dtype=np.uint8) * 200
            
            # Moving car 1
            x1 = 30 + frame_idx * 25
            if x1 < 550:
                cv2.rectangle(frame, (x1, 250), (x1+120, 340), (50, 50, 200), -1)
            
            # Moving car 2 (appears later)
            if frame_idx > 5:
                x2 = -50 + (frame_idx - 5) * 20
                if x2 < 500:
                    cv2.rectangle(frame, (x2, 280), (x2+100, 360), (200, 50, 50), -1)
            
            # Static person
            cv2.rectangle(frame, (450, 150), (520, 400), (200, 100, 50), -1)
            
            # Occasional dog (appears frame 8-15)
            if 8 <= frame_idx <= 15:
                cv2.rectangle(frame, (200, 350), (280, 430), (50, 200, 50), -1)
            
            # Run detection
            results = model(frame, conf=0.3, verbose=False)
            annotated = results[0].plot()
            
            # Count objects per class
            frame_counts = {}
            for box in results[0].boxes:
                cls_name = model.names[int(box.cls[0])]
                frame_counts[cls_name] = frame_counts.get(cls_name, 0) + 1
            
            frame_data.append({
                'frame': frame_idx,
                'counts': frame_counts,
                'total': sum(frame_counts.values())
            })
            annotated_frames.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        
        return frame_data, annotated_frames
    
    # Process video
    frame_data, annotated_frames = process_video_pipeline(20)
    
    # Extract time series
    frames = [d['frame'] for d in frame_data]
    total_counts = [d['total'] for d in frame_data]
    
    # Per-class time series
    all_classes = set()
    for d in frame_data:
        all_classes.update(d['counts'].keys())
    
    class_series = {cls: [d['counts'].get(cls, 0) for d in frame_data] for cls in all_classes}
    
    # Visualize
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # Sample frames
    sample_indices = [0, 5, 10, 15, 19]
    for idx, frame_idx in enumerate(sample_indices):
        ax = fig.add_subplot(gs[0, idx if idx < 3 else idx-3])
        if idx < 3:
            ax.imshow(annotated_frames[frame_idx])
            ax.set_title(f'Frame {frame_idx}', fontsize=10, fontweight='bold')
            ax.axis('off')
        else:
            ax.axis('off')
    
    # Fill remaining top slots
    for idx in range(2, 5):
        if idx >= 3:
            ax = fig.add_subplot(gs[0, idx-3])
            ax.imshow(annotated_frames[sample_indices[idx]])
            ax.set_title(f'Frame {sample_indices[idx]}', fontsize=10, fontweight='bold')
            ax.axis('off')
    
    # Total object count over time
    ax1 = fig.add_subplot(gs[1, :2])
    ax1.plot(frames, total_counts, marker='o', linewidth=2, markersize=6, color='navy')
    ax1.fill_between(frames, total_counts, alpha=0.3, color='lightblue')
    ax1.set_title('📈 Total Object Count Over Time', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Frame Number')
    ax1.set_ylabel('Object Count')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, max(total_counts) + 1)
    
    # Per-class breakdown
    ax2 = fig.add_subplot(gs[1, 2])
    colors = plt.cm.Set2(np.linspace(0, 1, len(class_series)))
    for (cls, counts), color in zip(class_series.items(), colors):
        ax2.plot(frames, counts, marker='s', label=cls, linewidth=2, color=color)
    ax2.set_title('📊 Per-Class Count', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Frame')
    ax2.set_ylabel('Count')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Summary statistics
    ax3 = fig.add_subplot(gs[2, :])
    ax3.axis('off')
    
    summary = "📹 Video Processing Pipeline Summary\n" + "="*50 + "\n\n"
    summary += f"🎬 Total frames processed: {len(frame_data)}\n"
    summary += f"📊 Average objects/frame: {np.mean(total_counts):.1f}\n"
    summary += f"🔺 Peak objects in frame: {max(total_counts)} (Frame {frames[total_counts.index(max(total_counts))]})\n"
    summary += f"🔻 Minimum objects in frame: {min(total_counts)}\n\n"
    
    summary += "📋 Per-Class Totals:\n"
    for cls, counts in class_series.items():
        total = sum(counts)
        avg = np.mean([c for c in counts if c > 0]) if any(counts) else 0
        summary += f"   • {cls}: {total} total detections, avg {avg:.1f} when present\n"
    
    summary += "\n💾 To save video: use cv2.VideoWriter() with annotated frames"
    
    ax3.text(0.05, 0.5, summary, fontsize=11, family='monospace',
            verticalalignment='center', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    
    plt.suptitle('✅ Solution 4: Video Processing Pipeline', fontsize=16, fontweight='bold')
    plt.show()
    
    # Save video code template
    save_code = '''
    # 💾 Save annotated video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter('output.mp4', fourcc, 30.0, (640, 480))
    for frame in annotated_frames:
        out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    out.release()
    '''
    print(save_code)

exercise_4_solution()

## 🎓 Summary & Day 88 Teaser <a id='12'></a>

### 🏆 What We Learned Today:
✅ **Object Detection fundamentals** — the leap from classification to localization  
✅ **YOLO architecture** — single-shot detection with incredible speed  
✅ **Pre-trained inference** — detecting 80+ COCO classes out-of-the-box  
✅ **Real-time processing** — images, videos, and FPS benchmarking  
✅ **Custom training pipeline** — dataset preparation, training curves, and fine-tuning  
✅ **Evaluation metrics** — mAP, Precision, Recall, IoU, and confusion matrices  
✅ **Deployment strategies** — edge, cloud, mobile, and industry applications  

### 🚀 Key Takeaways:
- YOLO processes images in **a single forward pass**, making it incredibly fast
- **Transfer learning** from COCO pre-trained weights dramatically reduces training time
- **mAP@0.5:0.95** is the gold standard metric for object detection evaluation
- Choosing the right **confidence threshold** balances precision and recall for your use case
- Model export to **ONNX/TensorRT** enables production deployment across platforms

---

### 🔮 Day 88 Teaser:
Tomorrow we dive into **Instance Segmentation with Mask R-CNN & YOLO-Seg**! We'll learn to predict pixel-level masks for each detected object, enabling precise object boundaries for applications like medical imaging, fashion segmentation, and autonomous driving lane detection. Get ready to go beyond bounding boxes! 🎭✨

---

> *"The best way to predict the future is to implement it."* — Keep building, keep learning! 🚀

**⭐ Day 87 Complete! See you tomorrow for Day 88! ⭐**